# Step 4 — CIFAR-FS Bertinetto split + episodic meta-training

**Spec:** `implementation.txt`, Step 4 (Phase 2).

**Goal:** Replace the CIFAR-100-test stand-in with the proper Bertinetto 2019 64/16/20 CIFAR-FS split, swap single-episode training for episodic meta-training (parameter-free prototype head, per-epoch validation, early stop), and produce the Phase A baseline numbers reported as mean ± 95% CI on 600 test-split episodes.

**Protocol decisions locked in (plan.txt §4 Phase 2):**
1. **ProtoNet-style** meta-training — 1 outer gradient step per episode, parameter-free prototype head, NO per-episode-from-scratch fine-tuning.
2. **5-shot training only** — 1-shot is deferred to the Step 10 MVT grid.
3. **Linear-head → prototype-head deviation** from proposal §5B is documented in Chapter 3 of the thesis.

**Exit criteria** (all must tick before closing Step 4):
- `data/cifar_fs_split.json` is the canonical Bertinetto split (not the synthetic fallback)
- both configs complete on Colab T4 in ≤ 30 min
- `results/phase2_bottleneck_prototype-{evidential,softmax}_metrics.json` exist
- R1 AUROC gap > 0.05 survives the move to real Bertinetto classes
- `pytest tests/` passes (31+ tests including the 3 new ones)
- same config twice → byte-identical metrics.json (Step 3 invariant)

## 0. Setup

Identical to Step 3 — clone the repo into Drive on Colab so the work persists across runtime restarts. On local the script walks up from CWD.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("In Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GITHUB_URL = "https://github.com/notAvailable73/thesis"

def _looks_like_repo(p: Path) -> bool:
    return (p / "src").is_dir() and (p / "configs").is_dir()

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        WORKDIR = "/content/drive/MyDrive/bpeft_step4"
    except Exception as e:
        print("Drive mount skipped:", e)
        WORKDIR = "/content/bpeft_step4"
    os.makedirs(WORKDIR, exist_ok=True)
    REPO = Path(WORKDIR) / "thesis"
    if not _looks_like_repo(REPO):
        print(f"Cloning {GITHUB_URL} into {REPO} ...")
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(REPO)], check=True)
    else:
        print(f"Reusing existing clone at {REPO}")
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    cur = Path.cwd().resolve()
    REPO = next((c for c in [cur, *cur.parents] if _looks_like_repo(c)), None)
    if REPO is None:
        raise RuntimeError("repo root not found (no src/ + configs/ above CWD)")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

## 1. Install dependencies (wandb)

Same as Step 3 — install wandb if missing.

In [ ]:
try:
    import wandb
    print("wandb already installed:", wandb.__version__)
except ImportError:
    print("installing wandb ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb>=0.16"], check=True)
    import wandb
    print("wandb installed:", wandb.__version__)

import yaml
print("pyyaml:", yaml.__version__)

## 2. wandb login

Put a 40+ character `WANDB_API_KEY` into Colab Secrets (Tools → Settings → Secrets). If absent or invalid, the notebook falls back to OFFLINE mode and you can `wandb sync wandb/` later.

In [ ]:
WANDB_MODE = "online"   # "online" | "offline" | "disabled"
api_key = None

if IN_COLAB and not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("WANDB_API_KEY")
        if api_key:
            os.environ["WANDB_API_KEY"] = api_key
            print("Loaded WANDB_API_KEY from Colab Secrets.")
    except Exception as e:
        print("Colab Secrets lookup skipped:", e)

if WANDB_MODE == "online":
    try:
        ok = wandb.login(key=api_key) if api_key else wandb.login()
        if not ok:
            print("wandb.login() returned False -- falling back to offline mode.")
            WANDB_MODE = "offline"
    except Exception as e:
        print(f"wandb.login() failed: {e!r}  -- falling back to offline mode.")
        WANDB_MODE = "offline"

print("WANDB_MODE for this session:", WANDB_MODE)

## 3. Pre-flight (verify the Step 1-3 foundation hasn't drifted)

Three checks:
1. `pytest tests/` still green (28 Step 1-3 tests + 3 new Step 4 tests = 31+).
2. `configs/test_episodes.yaml` has 600 seeds [0..599] (canonical truth from Step 3).
3. `configs/val_episodes.yaml` has 100 seeds [10000..10099] (new in Step 4), disjoint from the test list.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short"],
    capture_output=True, text=True, cwd=str(REPO),
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:\n", result.stderr[-1500:])
    raise SystemExit("pytest failed -- fix BEFORE starting Step 4")
print(f"Return code: {result.returncode}")

test_eps = yaml.safe_load(open(REPO / "configs/test_episodes.yaml"))
val_eps  = yaml.safe_load(open(REPO / "configs/val_episodes.yaml"))
assert test_eps["num_episodes"] == 600 and len(test_eps["seeds"]) == 600
assert test_eps["seeds"] == list(range(600))
assert val_eps["num_episodes"] == 100 and len(val_eps["seeds"]) == 100
assert val_eps["seeds"]  == list(range(10000, 10100))
assert set(test_eps["seeds"]).isdisjoint(set(val_eps["seeds"])), \
    "test and val seed ranges overlap!"
print("pre-flight OK: pytest passes; test_episodes (600) + val_episodes (100) disjoint.")

## 4. Fetch the canonical Bertinetto CIFAR-FS split

The repo ships `data/cifar_fs_split.json` with a SYNTHETIC fallback (alphabetical 0-63 / 64-79 / 80-99). This cell downloads the canonical 64/16/20 split and overwrites the fallback. Without this step, training silently uses the wrong classes; `load_cifar_fs_split()` warns when the file's `_status` is still `synthetic_fallback`.

**Source:** torchmeta's mirror at https://github.com/tristandeleu/pytorch-meta/tree/master/torchmeta/datasets/assets/cifar100/cifar-fs — three JSON files (`train.json`, `val.json`, `test.json`) each holding `[superclass, class_name]` pairs. We keep only the `class_name`. The original `bertinetto/r2d2` repo expects users to supply the split themselves and no longer hosts the lists.

**Fallback:** the same canonical class names are embedded in this cell as `EMBEDDED` so the notebook works offline. When the network fetch succeeds, the cell additionally asserts the fetched version matches `EMBEDDED` — a built-in canary for "torchmeta changed the split underneath us".

In [ ]:
import urllib.request

# Source: torchmeta's canonical CIFAR-FS asset files (the original
# bertinetto/r2d2 repo no longer hosts the split). The format is a JSON
# list of [superclass, class_name] pairs; we keep only the class_name.
# Verified URLs (via GitHub Contents API on master branch).
BASE_URL = ("https://raw.githubusercontent.com/tristandeleu/pytorch-meta/"
            "master/torchmeta/datasets/assets/cifar100/cifar-fs")
SPLIT_FILES = {
    "train": f"{BASE_URL}/train.json",
    "val":   f"{BASE_URL}/val.json",
    "test":  f"{BASE_URL}/test.json",
}


def _fetch_torchmeta_split(url: str) -> list[str]:
    with urllib.request.urlopen(url, timeout=30) as r:
        pairs = json.loads(r.read().decode("utf-8"))
    # Each entry is [superclass, class_name] — we want class_name only.
    return [pair[1] for pair in pairs]


# Embedded canonical Bertinetto CIFAR-FS split (verified against
# torchmeta's assets on 2026-05-20). Used as a fallback if the network
# fetch fails — and as belt-and-braces verification that the fetched
# version is correct.
EMBEDDED = {
    "train": [
        "dolphin", "seal", "aquarium_fish", "ray", "trout", "orchid",
        "sunflower", "tulip", "bottle", "bowl", "can", "cup", "plate",
        "apple", "mushroom", "orange", "pear", "clock", "keyboard",
        "chair", "couch", "bee", "caterpillar", "cockroach", "bear",
        "lion", "tiger", "wolf", "bridge", "castle", "house", "road",
        "skyscraper", "cloud", "forest", "mountain", "elephant",
        "kangaroo", "porcupine", "possum", "raccoon", "skunk", "lobster",
        "spider", "boy", "girl", "dinosaur", "lizard", "snake", "turtle",
        "hamster", "mouse", "rabbit", "shrew", "squirrel", "oak_tree",
        "palm_tree", "pine_tree", "willow_tree", "bus", "train",
        "lawn_mower", "streetcar", "tank",
    ],
    "val": [
        "beaver", "otter", "flatfish", "shark", "lamp", "television",
        "beetle", "butterfly", "sea", "camel", "cattle", "crab",
        "crocodile", "maple_tree", "motorcycle", "tractor",
    ],
    "test": [
        "whale", "poppy", "rose", "sweet_pepper", "telephone", "bed",
        "table", "wardrobe", "leopard", "plain", "chimpanzee", "fox",
        "snail", "worm", "baby", "man", "woman", "bicycle",
        "pickup_truck", "rocket",
    ],
}
assert len(EMBEDDED["train"]) == 64
assert len(EMBEDDED["val"])   == 16
assert len(EMBEDDED["test"])  == 20

try:
    canonical = {k: _fetch_torchmeta_split(v) for k, v in SPLIT_FILES.items()}
    print("fetched canonical split from torchmeta.")
    # Belt-and-braces: assert the fetched version matches our embedded copy.
    for k in ("train", "val", "test"):
        if sorted(canonical[k]) != sorted(EMBEDDED[k]):
            print(f"!!! WARNING: fetched {k} differs from embedded {k} -- "
                  f"using fetched version (torchmeta is authoritative). "
                  f"Update EMBEDDED to match.")
except Exception as e:
    print(f"!!! Fetch failed ({e!r}); falling back to embedded canonical split.")
    canonical = {k: list(v) for k, v in EMBEDDED.items()}

print("split sizes:", {k: len(v) for k, v in canonical.items()})
assert len(canonical["train"]) == 64
assert len(canonical["val"])   == 16
assert len(canonical["test"])  == 20

# Cross-check against torchvision's CIFAR-100 class names.
from src.datasets import CIFAR100_CLASS_NAMES
known = set(CIFAR100_CLASS_NAMES)
for split, names in canonical.items():
    bad = [n for n in names if n not in known]
    assert not bad, f"unknown CIFAR-100 names in {split}: {bad}"

# Disjoint and cover-all assertions.
train_s, val_s, test_s = (set(canonical[k]) for k in ("train", "val", "test"))
assert train_s.isdisjoint(val_s) and train_s.isdisjoint(test_s) and val_s.isdisjoint(test_s)
assert train_s | val_s | test_s == known

split_path = REPO / "data" / "cifar_fs_split.json"
blob = {
    "_comment": ("CIFAR-FS class split. Train/val/test classes from "
                 "CIFAR-100, encoded as class NAMES."),
    "_canonical_source": ("Bertinetto et al. 2019 split, mirrored at "
                          "https://github.com/tristandeleu/pytorch-meta/"
                          "tree/master/torchmeta/datasets/assets/cifar100/"
                          "cifar-fs"),
    "_status": "canonical_bertinetto_via_torchmeta",
    "_freeze": "FROZEN. Do not regenerate.",
    "train": sorted(canonical["train"]),
    "val":   sorted(canonical["val"]),
    "test":  sorted(canonical["test"]),
}
with open(split_path, "w") as f:
    json.dump(blob, f, indent=2, sort_keys=True)
print(f"wrote canonical split: {split_path}")
print("  train:", blob["train"][:5], "...")
print("  val:  ", blob["val"][:5], "...")
print("  test: ", blob["test"][:5], "...")

# Reload via the production loader to confirm structural assertions pass.
import importlib, src.datasets.cifar_fs as _cf
importlib.reload(_cf)
split_ids = _cf.load_cifar_fs_split()
print("split loaded OK (ID counts):",
      {k: len(v) for k, v in split_ids.items()})

## 5. Train both Phase 2 configs

Two configs:

| Config | Head | Interpretation | KL anneal |
|---|---|---|---|
| `configs/exp_phase2_evidential.yaml` | prototype | evidential | 0 → 0.5 over 1000 outer steps |
| `configs/exp_phase2_softmax.yaml`    | prototype | softmax    | (n/a)                          |

Both run 30 epochs of 100 episodes each on the Bertinetto TRAIN split (64 classes), with per-epoch validation on the VAL split (16 classes) using the 100 frozen val seeds. Early stop fires after 5 epochs of no improvement.

**Smoke-collapse guard (R-EPISODIC-COLLAPSE):** if val_acc after epoch 1 ≤ 0.25 (5-way random chance + nothing), the trainer aborts with `EpisodicCollapse`. This stops Colab burning GPU on a doomed run.

Filename convention: scripts/evaluate.py writes
  `results/{suffix}_{adapter}_{head_descriptor}_metrics.json`
where `head_descriptor = 'prototype-evidential' | 'prototype-softmax'` for Phase 2, so the two configs land in distinct files.

In [ ]:
CFG_EVIDENTIAL = "configs/exp_phase2_evidential.yaml"
CFG_SOFTMAX    = "configs/exp_phase2_softmax.yaml"
NUM_TEST_EPISODES = 600   # set to 50 for a quick smoke; 600 = full Phase 2

SUB_ENV = os.environ.copy()
SUB_ENV["WANDB_MODE"] = WANDB_MODE

def run(cmd, env=None):
    print(">>>", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True, env=env)
    out = r.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if r.returncode != 0:
        print("STDERR:\n", r.stderr[-2000:])
        raise RuntimeError(f"{cmd[0]} failed (rc={r.returncode})")
    return r

for cfg_path in [CFG_EVIDENTIAL, CFG_SOFTMAX]:
    print("\n" + "=" * 70)
    print(f"  config: {cfg_path}")
    print("=" * 70)
    run([sys.executable, "scripts/train.py",
         "--config", cfg_path,
         "--wandb-mode", WANDB_MODE], env=SUB_ENV)
    run([sys.executable, "scripts/evaluate.py",
         "--config", cfg_path,
         "--num-episodes", str(NUM_TEST_EPISODES),
         "--wandb-mode", WANDB_MODE,
         "--results-suffix", "phase2"], env=SUB_ENV)

## 6. Byte-identical reproducibility check (Step 3 invariant)

Re-run the evidential evaluation a second time with `--wandb-mode disabled` and assert the metrics JSON is byte-identical. Phase 2 must preserve the Step 3 reproducibility invariant.

In [ ]:
import filecmp

ev_metrics = REPO / "results/phase2_bottleneck_prototype-evidential_metrics.json"
assert ev_metrics.exists(), f"missing: {ev_metrics}"

ev_metrics_copy = REPO / "results/phase2_bottleneck_prototype-evidential_metrics.PRIOR.json"
shutil.copy(ev_metrics, ev_metrics_copy)
print(f"First run    : {ev_metrics_copy} ({ev_metrics_copy.stat().st_size} bytes)")

# Re-run evaluation with wandb disabled (so we don't add a duplicate run to the dashboard).
run([sys.executable, "scripts/evaluate.py",
     "--config", CFG_EVIDENTIAL,
     "--num-episodes", str(NUM_TEST_EPISODES),
     "--wandb-mode", "disabled",
     "--results-suffix", "phase2"], env=SUB_ENV)
print(f"Second run   : {ev_metrics} ({ev_metrics.stat().st_size} bytes)")

identical = filecmp.cmp(ev_metrics, ev_metrics_copy, shallow=False)
assert identical, (
    f"metrics.json drifted between Phase 2 runs!\n"
    f"first : {ev_metrics_copy.read_text()[:400]}\n"
    f"second: {ev_metrics.read_text()[:400]}"
)
print("\nbyte-identical OK -- Step 3 reproducibility invariant preserved in Phase 2.")
ev_metrics_copy.unlink()

## 7. Sanity-check the numbers vs Step 3

Step 3 (stand-in CIFAR-100 test, classes 0..19) reported:

| Metric | Evidential | Softmax | Gap (E − S) |
|---|---|---|---|
| accuracy | 0.849 ± 0.006 | 0.823 ± 0.006 | +0.026 |
| ECE mean | 0.162 | 0.136 | +0.026 (softmax better) |
| OOD AUROC | 0.951 | 0.850 | **+0.101** |

Phase 2 numbers will be different (different test classes, different protocol). The success criterion is **AUROC gap > 0.05**. If that holds, the R1 story carries over to the real Bertinetto split and Phase 2 closes.

In [ ]:
ev = json.load(open(REPO / "results/phase2_bottleneck_prototype-evidential_metrics.json"))
sm = json.load(open(REPO / "results/phase2_bottleneck_prototype-softmax_metrics.json"))

def show(name, m):
    print(f"\n[{name}]   {m['num_episodes']} episodes  best_val_epoch={m.get('best_val_epoch', '?')}")
    print(f"  accuracy  : {m['accuracy_mean']:.3f} +/- {m['accuracy_ci95']:.3f}  (95% CI)")
    print(f"  Macro-F1  : {m.get('f1_macro_mean', 0.0):.3f}")
    print(f"  ECE mean  : {m['ece_per_episode_mean']:.3f}")
    print(f"  ECE pool  : {m['ece_pooled']:.3f}")
    print(f"  Brier     : {m['brier_mean']:.3f}")
    print(f"  OOD AUROC : {m['ood_auroc_mean']:.3f}")
    print(f"  FPR @95   : {m.get('fpr_at_95_tpr_mean', 1.0):.3f}")

show("phase2 evidential (prototype, KL anneal 0->0.5)", ev)
show("phase2 softmax    (prototype, no KL)",            sm)

auroc_gap = ev["ood_auroc_mean"] - sm["ood_auroc_mean"]
print(f"\nOOD AUROC gap (evidential - softmax): {auroc_gap:+.3f}")
if auroc_gap > 0.05:
    print("OK -- R1 trade-off survives the move to real Bertinetto split. Phase 2 closes.")
else:
    print("!!! R1 AUROC win did NOT survive the move to the real split.")
    print("!!! ESCALATE to supervisor before opening Step 5. See implementation.txt")
    print("!!! Step 4 exit criteria and the R-FALLBACK protocol note.")

## 8. Display final artifacts

Each evaluation produced three PNGs (reliability / OOD histogram / confusion matrix) and one JSON. The PNGs were also uploaded to W&B.

In [ ]:
from IPython.display import Image, display, Markdown

for tag in ["phase2_bottleneck_prototype-evidential",
            "phase2_bottleneck_prototype-softmax"]:
    for kind in ["reliability", "ood_histogram", "confusion_matrix"]:
        p = REPO / f"results/{tag}_{kind}.png"
        if p.exists():
            display(Markdown(f"**{tag} — {kind}**"))
            display(Image(filename=str(p)))

## 9. wandb dashboard links

Online mode: list this session's Phase 2 runs from the W&B API. Offline mode: point at the local `wandb/` dir to sync later.

In [ ]:
project = "bpeft-thesis"
if WANDB_MODE == "online":
    try:
        api = wandb.Api()
        runs = api.runs(f"{project}",
                        filters={"tags": {"$in": ["phase2", "step4"]}},
                        per_page=20)
        print("Phase 2 (Step 4) runs in this project:")
        for r in runs:
            print(f"  - {r.name}  ({r.state})  {r.url}")
    except Exception as e:
        print(f"Could not list runs via API ({e!r}). Check the dashboard manually at:")
        print(f"   https://wandb.ai/<your-entity>/{project}")
elif WANDB_MODE == "offline":
    print("Offline mode: artifacts written under ./wandb/. Run `wandb sync wandb/`")
    print("to upload them after the fact.")
else:
    print("wandb was disabled for this session.")

## 10. Exit-criteria checklist (mirrors progress.txt Step 4)

| Criterion | Where verified |
|---|---|
| `data/cifar_fs_split.json` is canonical Bertinetto (not synthetic fallback) | §4 |
| disjoint splits + cover all 100 CIFAR-100 classes | §4 + `tests/test_cifar_fs_split.py` |
| `scripts/train.py --config configs/exp_phase2_evidential.yaml` runs in ≤ 30 min | §5 |
| same for `configs/exp_phase2_softmax.yaml` | §5 |
| `results/phase2_bottleneck_prototype-*_metrics.json` exist with all required fields | §7 |
| R1 AUROC gap > 0.05 on real Bertinetto split | §7 (the assertion is the test) |
| same config twice → byte-identical `metrics.json` | §6 |
| `pytest tests/` passes (31+ tests) | §3 |

If every row is checked, Step 4 closes. Update `progress.txt` Step 4 actions to `[x]`, write `step_writeups/step4.txt`, and open Step 5 (LoRA + BitFit + non-PEFT baselines).